## Using edgeR for DEG analysis between clusters -- comparing region_leiden clusters within capillary cells and using pseurod-bulk approach

In [ ]:
### identify cluster marker genes
suppressPackageStartupMessages({
    library(Seurat)
    library(stringr)
    library(dplyr)
    library(patchwork)
    library(ggplot2)
    library(future)
    library(tidydr)
    library(harmony)
    library(reticulate)
    library(viridis)
    library(RColorBrewer)
    library(ComplexHeatmap)
    library(colorRamp2)
    library(edgeR)
})

# set large memory  
options(future.globals.maxSize = 120*1024^3)

## plotting parameters
umap_theme <- theme_dr()+theme(panel.grid.major = element_blank(), 
                                            panel.grid.minor = element_blank(),
                                            panel.background = element_blank(), 
                                            axis.line = element_line(colour = "black"))

## set working dir
dir = "/scratch/users/aliu10/vascular"
setwd(dir)

In [ ]:
# obj_oi = readRDS("./Results/Revision_2/Endothelial.rds")
## Reloading 
# Convert h5ad to h5seurat
h5ad_file="./Results/Revision_2/Pericyte_temp_processed.h5ad"
obj_oi = schard::h5ad2seurat(h5ad_file)
print(obj_oi)

In [ ]:
### optional: check how many brain regions are there
length(unique(obj_oi$brain_region))
table(obj_oi$brain_region,obj_oi$region_leiden)

In [ ]:
meta = obj_oi@meta.data
meta$region_cluster = gsub('Cluster_','C', meta$region_cluster)
obj_oi@meta.data = meta
table(obj_oi$region_cluster,useNA = "always")

## Check the UMAP of the mural cell types


In [ ]:
options(repr.plot.height = 7,repr.plot.width = 8)
p = DimPlot(obj_oi,
    group.by = "brain_region",
    # cols = col1,
    reduction = "umapharmony_",
    pt.size = 2.5,
    label = TRUE,
    label.size = 5,
    repel = TRUE
    ) + umap_theme + NoLegend()

print(p)

In [ ]:
## exclude samples with less than 3 cells
meta = obj_oi@meta.data
mat = table(meta$sampleID)
sample_excluded = names(mat[mat<3])
length(sample_excluded)

obj_oi = subset(obj_oi,subset = !sampleID %in% sample_excluded)
obj_oi

In [ ]:
## further filter genes with less expression
min_features_pct =0.05
min_features_ncell = 20

##--- Get the expression summary ---##
# obj_oi <- JoinLayers(obj_oi)
# expr = LayerData(obj_oi, assay = "decontX", layer = "counts")
expr = LayerData(obj_oi, assay = "RNA", layer = "counts")

## calculate percentage of cells expressing each gene
genes.percent.expression <- rowMeans(expr>0)
# genes.percent.expression = rowSums(expr>0)

## select genes based on expression threshold
genes = names(genes.percent.expression[genes.percent.expression>min_features_pct])
# genes = names(genes.percent.expression[genes.percent.expression>min_features_ncell])

print(paste(length(genes),"genes will be tested."))

## Using wilcoxon rank sum test to identify cluster marker genes

In [ ]:
# Normalize the data
obj_oi <- NormalizeData(obj_oi)
Idents(obj_oi) = obj_oi$region_cluster
obj_oi

In [ ]:
all.markers <- FindAllMarkers(object = obj_oi, 
                              assay = "RNA",
                              features = genes,
                              only.pos = FALSE, 
                              min.pct = 0.25
                              )
all.markers %>% 
    group_by(cluster) %>% 
    filter(p_val_adj < 0.05, avg_log2FC > 0.20) %>%
    top_n(n = 5, wt = avg_log2FC) -> top

all.markers %>% 
    group_by(cluster) %>% 
    filter(p_val_adj < 0.05, avg_log2FC > 0) -> sig

In [ ]:
options(repr.plot.height = 4,repr.plot.width = 6)
p1 <-  DotPlot(obj_oi,cols = viridis(3, option = "magma"),
             features = unique(top$gene)) + RotatedAxis(180) +
                                    theme(axis.title = element_blank(),legend.position = "top")
p1

## Preparing for edgeR analysis

In [ ]:
## exclude samples with less than 3 cells
meta = obj_oi@meta.data
mat = table(meta$sampleID)
sample_excluded = names(mat[mat<3])
length(sample_excluded)

obj_oi = subset(obj_oi,subset = !sampleID %in% sample_excluded)
obj_oi

## Get psudo-bulk expression

In [ ]:
## getting the pseudo-bulk object
pseudo_obj <- AggregateExpression(obj_oi, assays = "RNA", return.seurat = T, group.by = c("sampleID","region_cluster"))
pseudo_obj

## getting the count matrix for edgeR analysis
mat = pseudo_obj[['RNA']]$counts
head(mat)

## getting the meta data for edgeR analysis
meta = pseudo_obj@meta.data
group = meta$region_cluster
head(meta)

In [ ]:
## Build edgeR format
y <- DGEList(counts=mat,group=group)
cpm_mat <- cpm(y, log=F)

## add donor info
df = str_split_fixed(rownames(y$samples),"-",4)[,c(1,2)]
y$samples$donor = paste(df[,1],df[,2],sep = "_")

## filter sample by number of reads
summary(y$samples$lib.size)

## Removing the samples with low number of reads lower than 5% quantile
cutoff <- quantile(y$samples$lib.size, 0.05)
print(cutoff)
keep.samples <- y$samples$lib.size > cutoff 
table(keep.samples)
y <- y[, keep.samples]
cpm_mat = cpm_mat[, keep.samples]

## preprocessing for model fitting
group <- droplevels(factor(y$samples$group))
donor <- droplevels(factor(y$samples$donor))

design <- model.matrix(~0 + group + donor)
colnames(design) <- make.names(colnames(design))
head(design)

## filter genes based on design
keep <- filterByExpr(y, design=design)
table(keep)
y <- y[keep, , keep.lib.sizes=FALSE]

In [ ]:
## normalize library sizes
y <- normLibSizes(y)
summary(y$samples$norm.factors)

options(repr.plot.width=12,repr.plot.height=12)
brain_region <- as.factor(y$samples$group)
plotMDS(y, pch=16, col=c(1:36)[y$samples$group], main="brain region",labels = y$samples$donor)

In [ ]:
## estimate dispersion
y <- estimateDisp(y, design, robust=TRUE)
##--------- fit the model -----------##
fit <- glmQLFit(y, design, robust=TRUE)

In [ ]:

K <- nlevels(group)
grp_cols <- grep("^group", colnames(design), value=TRUE)

# sanity: length(grp_cols) should be K
stopifnot(length(grp_cols) == K)

C <- matrix(-1/(K-1), nrow=K, ncol=K,
            dimnames=list(grp_cols, levels(group)))
diag(C) <- 1

# expand to full design rows (add zeros for donor columns)
Cfull <- matrix(0, nrow=ncol(design), ncol=K,
                dimnames=list(colnames(design), colnames(C)))
Cfull[grp_cols, ] <- C

qlf <- vector("list", K)
for(i in seq_len(K)) {
  qlf[[i]] <- glmQLFTest(fit, contrast=Cfull[, i])
  qlf[[i]]$comparison <- paste0(levels(group)[i], "_vs_others")
}


In [ ]:
res = data.frame()
for (i in 1:length(qlf)){
    df = qlf[[i]]$table
    df$genes = rownames(df)
    df$FDR = p.adjust(df$PValue, method = "bonferroni")
    df %>%
        dplyr::arrange(FDR) %>%
        # dplyr::filter(FDR  < 0.05) %>%
        # dplyr::filter(logFC > 0) %>% 
        ungroup() -> sig

    if (nrow(sig) == 0) next    
    
    sig$cluster = levels(group)[i]
    res = rbind(res,sig)
}

In [ ]:
res$direction = ifelse(res$logFC > 0, "up", "down")
table(res$cluster,res$direction)

In [ ]:
## write the significant results
write.csv(res,file = "./Results/Revision_2/DEG/Pericyte_rg_clusters_edgeR.csv")
# write.csv(res,file = "./Results/Revision_2/DEG/Endo_rg_clusters_edgeR_woC5.csv")

## Draw Heatmap of top DE genes

In [ ]:
## optional: reloading the res
res = read.csv("./Results/Revision_2/DEG/Pericyte_rg_clusters_edgeR.csv", row.names = 1)
table(res$cluster,res$direction)

In [ ]:
## Get the top 10 significant genes for each cluster
res %>%
  filter(logFC > 0.25) %>%
  filter(FDR < 0.05) %>%
  group_by(cluster) %>%
  # arrange(desc(logFC)) %>%
  arrange(FDR) %>%
  slice_head(n = 5) -> top


g_remove = c('STEAP1B', 'SLC7A14-AS1', 'SHTN1', 'SLC5A11')
top = top[!top$genes %in% g_remove,]
top
# write.csv(top,file = "./Results/Revision_2/DEG/Endo_rg_clusters_edgeR_top20_wC5.csv")

In [ ]:
### Making sure get the corrected layer and matrix
DefaultAssay(obj_oi) = "RNA"

### Aggregation to pseudobulk 
mtx = AggregateExpression(
        obj_oi, 
        # features = gene_oi,
        group.by = c("region_cluster","individualID"),
        assays = 'RNA',
        slot = "counts"
    ) 
mtx <- as.matrix(mtx$RNA)
## Get the library size for each sample
lib_size <- Matrix::colSums(mtx)

### Get the logCPM from the matrix
cpm <- t(t(mtx) / lib_size) * 1e6
logCPM <- log2(cpm + 1)
logCPM = logCPM[top$genes,]

### Calculate the averaged logCPM
# extract region (everything before first "_")
region <- sub("_.*$", "", colnames(logCPM))

# sanity check
table(region)

logCPM_region <- sapply(
  split(seq_len(ncol(logCPM)), region),
  function(i) {
    rowMeans(logCPM[, i, drop = FALSE])
  }
)
logCPM_region_z <- t(scale(t(logCPM_region)))

In [ ]:
## working on the top regulated first
df_all = res

# ## construct the matrix
# mat<-matrix(nrow=length(unique(top$genes)), ncol=length(unique(df_all$cluster)))
# colnames(mat) <- unique(df_all$cluster)

# ### create the matrix to show the gene expression change pattern
# for (j in 1:length(unique(top$genes))){
#     for (i in 1:length(unique(df_all$cluster))){
#         geneTemp<-unique(top$genes)[j]
#         rgTemp<-colnames(mat)[i]

#         sub<-df_all[which(df_all$genes==geneTemp & df_all$cluster==rgTemp),]
#         if (nrow(sub)>0){
#         mat[j,i]<-sub$logFC
#         }
#         else{
#         mat[j,i]<-0
#         }
#     }  
# }
# rownames(mat)<- unique(top$genes)
mat = logCPM_region_z

#generating a significance matrix
sig_mat<-matrix(nrow=length(unique(top$genes)), ncol=length(unique(df_all$cluster)))
colnames(sig_mat)<-unique(df_all$cluster)

for (j in 1:length(unique(top$genes))){
    for (i in 1:length(unique(df_all$cluster))){
        geneTemp<-unique(top$genes)[j]
        rgTemp<-colnames(sig_mat)[i]
        sub<-df_all[which(df_all$genes==geneTemp & df_all$cluster==rgTemp),]
        if (nrow(sub)>0){
            sig_mat[j,i]<-sub$FDR
        }else{
        sig_mat[j,i]<-1
        }
    }
}
rownames(sig_mat)<- unique(top$genes)

## reorder the matrix
mat <- mat[,order(as.numeric(str_split_fixed(colnames(mat),pattern = "_",2)[,1]))]
sig_mat <- sig_mat[,order(as.numeric(str_split_fixed(colnames(sig_mat),pattern = "_",2)[,1]))]
# mat
# sig_mat

In [ ]:
# pdf("./Results/Revision_2/DEG/Figure_2X_Endothelial_DEG_heatmap_wC5_1.pdf",width = 4,height = 12)
ht <- Heatmap(
  mat,
  cluster_rows    = FALSE,
  cluster_columns = FALSE,
  col = colorRamp2(c(-1,0,1), c("#5387be","#F7F7F7","#b31f2c")),
  # right_annotation = row_ha,
  show_column_names = TRUE,
  show_row_names    = TRUE,
  column_title      = NULL,
  row_names_side    = "left",
  row_title_gp      = gpar(fontsize = 10),
  row_names_gp      = gpar(fontsize = 8),
  row_names_max_width = max_text_width(rownames(mat), gp = gpar(fontsize = 12)),
  cell_fun = function(j, i, x, y, w, h, fill) {
    if (sig_mat[i, j] < 0.05)
      grid.text("*", x, y, gp = gpar(fontsize = 18, col = "black"), vjust = "center")
  }
)

pdf("./Results/Revision_2/DEG/Figure_4X_Pericyte_DEG_heatmap.pdf",width = 3,height = 4)
draw(ht)
dev.off()

### Enrichment analysis for the sig genes between clusters and others


In [ ]:
res = read.csv("./Results/Revision_2/DEG/Pericyte_rg_clusters_edgeR.csv",row.names = 1)
res_sig = res %>% dplyr::filter(FDR < 0.05) %>% dplyr::filter(logFC > 0.25)
table(res_sig$cluster)
# res

In [ ]:
suppressPackageStartupMessages({
library(clusterProfiler)
library('org.Hs.eg.db')
# library(ReactomePA)
})

In [ ]:
#########################################################################
#### find enriched GOBP for consistent signals in inhibitory neurons ####
#########################################################################
res$gene_id <- mapIds(
  # Replace with annotation package for the organism relevant to your data
  org.Hs.eg.db,
  # The vector of gene identifiers we want to map
  keys = res$genes,
  # Replace with the type of gene identifiers in your data
  keytype = "SYMBOL",
  # Replace with the type of gene identifiers you would like to map to
  column = "ENTREZID",
  multiVals = "first"
)

In [ ]:
### Run enrichment analysis for overlapped upregulated genes in C1, C2, and C3 clusters
sig_genes_list = res %>%
  dplyr::filter(FDR < 0.05) %>%
  dplyr::filter(logFC > 0.25) %>%
  group_by(cluster)
sig_genes_list = split(sig_genes_list$gene_id, sig_genes_list$cluster)
sig_genes_list = lapply(sig_genes_list, unique)
sig_genes_list = sig_genes_list[sapply(sig_genes_list, length) > 0]
length(sig_genes_list)

# # ## get the common upregulated genes in C1, C2, and C3 clusters
# # common_genes = Reduce(intersect, sig_genes_list[c("C2",'C3','C1')])
# # length(unique(common_genes))

In [ ]:
cluster_genes <- res %>%
    dplyr::filter(cluster == "C3", logFC > 0.25, FDR < 0.05) %>%
        pull(gene_id) %>%
            unique()
ego <- enrichGO(gene = cluster_genes,
            OrgDb = org.Hs.eg.db,
            keyType = "ENTREZID",
            ont = "BP",
            pAdjustMethod = "BH",
            readable      = TRUE,
            pvalueCutoff = 0.05,
            qvalueCutoff = 0.05)

In [ ]:
mutate(ego, qscore = -log(p.adjust, base=10)) |> 
    barplot(x="qscore")

In [ ]:
sessionInfo()